# IMDb Top Netflix Movies and TV Shows

Dataset link: https://www.kaggle.com/datasets/bharatnatrayn/movies-dataset-for-feature-extracion-prediction?select=movies.csv

## Contents
- [Imports and Loading](#imports-and-loading)
- [Overview](#overview)
  - [Whitespace and Column Cleaning](#whitespace-and-column-cleaning)
  - [Duplicate Handling](#duplicate-handling)
  - [Rejected records creation](#rejected-records-creation)
  - [Before and After records](#before-and-after-records)
  - [Overall insights](#overall-insights)
- [Data Fromatting and Type Conversion](#data-fromatting-and-type-conversion)
  - Numerical data conversion
  - [Formatting strings](#formatting-strings)
  - [Conclusion](#conclusion)
- [ROMI Approach](#romi-approach)
  - [Relation](#relation)
  - [Outlier](#outlier)
    - Investigating `Column`
  - [Mismatch](#mismatch)
  - [Interpolation](#interpolation)
- [Final Check](#final-check)
- [Before VS After](#before-vs-after)
- [Export](#export)

## Imports and Loading

In [18]:
import pandas as pd
import re

In [19]:
df = pd.read_csv("data/raw.csv")

print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded dataset: 9999 rows, 9 columns


## Overview

In [20]:
df.head()

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross
0,Blood Red Sky,(2021),"\nAction, Horror, Thriller",6.1,\nA woman with a mysterious illness is forced ...,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,NaN
1,Masters of the Universe: Revelation,(2021– ),"\nAnimation, Action, Adventure",5.0,\nThe war for Eternia begins again in what may...,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,NaN
2,The Walking Dead,(2010–2022),"\nDrama, Horror, Thriller",8.2,\nSheriff Deputy Rick Grimes wakes up from a c...,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,NaN
3,Rick and Morty,(2013– ),"\nAnimation, Adventure, Comedy",9.2,\nAn animated series that follows the exploits...,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,NaN
4,Army of Thieves,(2021),"\nAction, Crime, Horror",NaN,"\nA prequel, set before the events of Army of ...",\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,NaN


### Whitespace and Column Cleaning

In [21]:
# whitespace cleaning
cols = df.select_dtypes(include=["object", "str", "string"]).columns.tolist()

for col in cols:
    df[col] = df[col].str.replace(r"[\t\n]+", " ", regex=True).str.replace(r"\s{2,}", " ", regex=True).str.strip()

In [22]:
# column cleaning
print("Initial columns:", df.columns.tolist())
df.columns = df.columns.str.strip().str.lower().str.replace("-", "_")
print("Clean columns:", df.columns.tolist())

Initial columns: ['MOVIES', 'YEAR', 'GENRE', 'RATING', 'ONE-LINE', 'STARS', 'VOTES', 'RunTime', 'Gross']
Clean columns: ['movies', 'year', 'genre', 'rating', 'one_line', 'stars', 'votes', 'runtime', 'gross']


### Duplicate Handling

In [23]:
print("(1) Duplicates:", df.duplicated(subset=["movies", "one_line"]).sum())
display(df[df.duplicated(subset=["movies", "one_line"], keep=False)].sort_values(by="movies").head(10))

(1) Duplicates: 857


,movies,year,genre,rating,one_line,stars,votes,runtime,gross
9992,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9989,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9990,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9443,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar | Stars: Aneurin Barna...,NaN,NaN,NaN
9991,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9982,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9980,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9981,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9177,800 metros,(2021– ),Documentary,NaN,Add a Plot,Director: León Siminiani,NaN,NaN,NaN
9178,800 metros,(2021– ),Documentary,NaN,Add a Plot,Director: León Siminiani,NaN,NaN,NaN


*Question 1 for the client*

---

Hi,

I noticed the dataset doesn’t have a unique ID column. This makes it difficult to confidently identify duplicates.

In addition, there are no indications of what type the record is (Movie/Series). And if series, no season and episode number. This makes it even more difficult to say if two identical records are duplicates or different episodes of the same TV series. Hence, **I will proceed by assuming there are no duplicate rows and treating identical records distinctively**. Though, I’ll make sure everything is document clearly.

Is this approach fine with you?